# OBPS Data Processing

Processing OBPS data files into MESSAGE-ix format

In [1]:
# Read Excel files and import relevant packages

import pandas as pd
import os
import numpy as np
import re

# Sys and OS Operations
os.chdir("../../")




In [2]:
import os

print(os.getcwd())


c:\Users\brend\Documents


In [9]:
input_file = "github/message-ix - CA/data/MESSAGE_OBPS_stringent.csv"
read_file = pd.read_csv(input_file, sep=',')

mapping = pd.read_excel("github/message-ix - CA/data/obps_inputs_clean.xlsx", sheet_name="Mapping")

#mapping["variable"] = mapping["variable"].str.split("|")[3:].aggregate(lambda x: "|".join(x))


# Data Viewer Cell

In [ ]:
mapping
#read_file

# Final Dataframe - Mapping Technologies

Created dictionaries for unit conversions

>Note: densities are recorded in tonnes per metre cubed

* Conversion of different commodity volumes to GJ
    1. For "oil", "loil", "ref_hil", and "foil" products: used heavy crude oil value
    2. For "oil_extr" products: used bitumen value (m3 to GJ)
    3. For "coal" and "igcc" products: used bitumenous coal value
    4. For "gas" products: assumed natural gas plant


In [10]:
# Create dataframe that is a copy of the original dataframe, but with the corrsponding technologies in a "technology column"

mapping_dict = mapping.groupby('variable')['technology'].apply(list).to_dict()
new_units_dict = dict(zip(read_file["unit [tCO2/-]"].unique(), ["tonne", "tonne", "MWh", "GWh", "tonne", "GWh", 
                                                                "CAN-CWB", "m3", "barrel_bitumen", "tonne_h2", 
                                                                "MWh", "m3", "AB_index", "100e3_m3", "1e3_AB-CWB", 
                                                                "CAN-CWB", "barrel_heavy_crude", "-"]))

#-#-#-#-#-#-#-#-#-#-#-#-#-#-#-#-#-#-#-#-#-#-#-#-#-#-#-#-#-#

# Iterate through the main dataframe and update 'technology' where a match is found
def tech_mapping(variable):
    techs = []
    for var, tech in mapping_dict.items():
        if var in variable:
            techs.extend(tech)

    try:
        return techs
    except Exception as e:
        return "No associated technologies"

def convert_to_KWa(data):

    # Conversion of different commodity volumes to GJ
        ## For "loil", "ref_hil", and "foil" products: used heavy crude oil value (m3 to GJ)
        ## For "oil_extr" products: used bitumen value (m3 to GJ)
        ## For "coal" and "igcc" products: used bitumenous coal value (tonnes to GJ)
        ## For "gas" products: assumed natural gas plant (m3 to GJ)
        
    # Densities in tonnes per m3
    general_conversion_dict = {"GJ_to_Kwa": 0.03171, "methanol_density": 0.7914, "barrel_to_m3": 0.159, 
                               "h2_density": 0.09e-3}
    
    # Conversion of different commodity volumes to GJ
    tech_to_GJ_dict = {"meth_": 15.6, "liq_": 23.6, "gas_": 0.0373, "loil_": 40.9,
                       "foil_": 40.9, "coal_": 27.6, "igcc": 27.6, "h2": 0.012, "ref_": 40.9,
                       "oil_extr": 42.8, "syn_liq": 38.51}
    
    all_techs = data["technology"].unique()
    all_techs_to_GJ = {}
    
    for tech in all_techs:
        for prefix, rate in tech_to_GJ_dict.items():
            if tech.startswith(prefix):
                all_techs_to_GJ[tech] = rate
        
    # Initialize converted_values list
    converted_values = []
    
    # Process some conversions (vectorized)
    data.loc[data["unit [tCO2/-]"] == "barrel_bitumen", "value"] *= general_conversion_dict["barrel_to_m3"]
    data.loc[data["unit [tCO2/-]"] == "100e3_m3", "value"] /= 100e3
    data.loc[data["unit [tCO2/-]"] == "1e3_AB-CWB", "value"] /= 1e3
    


    # Apply mask to apply conversion only to standards
    data["conversion_to_KWa"] = data["value"]
    mask_stds = data["variable"].str.contains("Standard", na=False)
    data_stds = data.loc[mask_stds,:]
    
    # Process the technologies and apply the GJ conversion
    for tech, conversion_factor in all_techs_to_GJ.items():
        mask_techs = data_stds["technology"] == tech
        data_stds.loc[mask_techs, "conversion_to_KWa"] = data_stds.loc[mask_techs, "value"] * conversion_factor * general_conversion_dict["GJ_to_Kwa"]
        #converted_values.append(data_stds.loc[mask_techs, "conversion_to_KWa"])
    '''
    for conv_val in converted_values:
        data["conversion_to_KWa"] = conv_val 
    ''' 
    # Final conversion to KWa
    #data["conversion_to_KWa"] *=  general_conversion_dict["GJ_to_Kwa"]
    
    # Return the updated values
    data["conversion_to_KWa"] = data_stds["conversion_to_KWa"]
    # Ignore rows with tightening rates
    data["conversion_to_KWa"] = data["conversion_to_KWa"].fillna(0)
    return data
    #return data["conversion_to_KWa"].tolist()
            
    

#-#-#-#-#-#-#-#-#-#-#-#-#-#-#-#-#-#-#-#-#-#-#-#-#-#-#-#-#-#
#mapping_dict = dict(zip(mapping['variable'].astype(str), mapping['technology'].astype(str)))
#read_file['technology'] = read_file['variable'].astype(str).map(mapping_dict)

# Expand technology column with new rows for each technology
read_file["technology"] = read_file["variable"].apply(lambda x: tech_mapping(x))
read_file = read_file.explode("technology")
read_file = read_file[read_file["technology"].notna()]

read_file["unit [tCO2/-]"] = read_file["unit [tCO2/-]"].replace(new_units_dict)

read_file.reset_index(drop=True, inplace= True) # Reset index after operations

read_file = convert_to_KWa(read_file.copy())
read_file["comments"] = pd.NA # Add comments column


# Write values to file
read_file.to_csv("github/message-ix - CA/data/New_OBPS_stringent.csv", index=False, header=True)

'''# Write to Excel
output_file_path = "github/message-ix - CA/data/New_OBPS_Values.xlsx"
read_file.to_excel(output_file_path, index=False)'''



'# Write to Excel\noutput_file_path = "github/message-ix - CA/data/New_OBPS_Values.xlsx"\nread_file.to_excel(output_file_path, index=False)'

In [8]:
#read_file["technology"].unique()
read_file["unit [tCO2/-]"].unique()

KeyError: 'unit [tCO2/-]'

# Test Model

In [ ]:
from model.Model import model_MSG

msg = model_MSG(scenario_name='baseline', config=False)
platform, msg_scenario, msg_folder = msg.setup_scenario()